<div style="background: linear-gradient(135deg, #e8f4f8 0%, #c4dce8 100%); padding: 30px 32px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: #1a3a4a; margin: 0; font-size: 28px;">E10 &middot; Day 3 &middot; Tool Calling &amp; MCP &mdash; Let the Model Reach Your Systems</h1>
<p style="color: #3a6a8a; margin: 8px 0 0 0; font-size: 16px;">GenAI for Engineering Managers &mdash; Exercise 10 of 15 &middot; Facilitator-run (watch, or try alongside)</p>
<p style="color: #2a4a5a; margin: 14px 0 0 0; font-size: 14px; line-height: 1.6;">
<strong>Why this exercise:</strong> In E09 we could finally query our own retail data &mdash; but notice who did the querying: <em>we</em> did. A human wrote the question, a human ran the lookup, a human pasted the result back to the model. The model itself still cannot touch a single live system. This session closes that gap: first the honest failure (a model asked about a live shipment, with no way to check), then <strong>tool calling</strong> &mdash; the mechanism that lets a model request a real function call &mdash; and <strong>MCP</strong>, the open standard that turns those functions into a plug-and-play catalog any AI application can discover. You will leave able to ask a team two sharp questions: <em>"which tools does the model get?"</em> and <em>"who decided that list?"</em>
</p>
</div>

<hr class='section-divider'>

## Part 1 &nbsp;&middot;&nbsp; The Gap E09 Left Open

<div class='topic-header'>
<strong>Where we are.</strong> Day 3 so far: E08 gave the assistant our <em>documents</em> (RAG), E09 gave it our <em>structured data</em> (SQL over the retail database). Both are <strong>reads of a snapshot</strong>, orchestrated by us. Two things are still impossible:
<br><br>
<strong>1. Data that changes by the minute.</strong> Where is the replenishment shipment for store 4479 <em>right now</em>? A vector index or a report is stale the moment a trailer moves.<br>
<strong>2. Actions.</strong> RAG can quote the emergency-transfer policy; it cannot <em>create</em> an emergency transfer. Reading is not doing.
</div>

<div class='concept-box'>
The demo domain today: <strong>inbound shipment tracking</strong> &mdash; the question every store operations manager asks the supply-chain team daily: <em>"where is my shipment?"</em> The live system of record is a small file, <code>data/shipments.json</code>, standing in for the transportation-management API. Everything you see generalises: swap the file for the real API and the pattern is identical.
</div>

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Facilitator note: the key below is set live during the session (same as E01).
import openai, json
from pathlib import Path

client = openai.OpenAI(
    api_key='PASTE_THE_KEY_SHARED_IN_SESSION_HERE'
)
MODEL = 'gpt-5.4-nano'

SHIPMENTS_FILE = Path('..') / 'data' / 'shipments.json'
_original_shipments = SHIPMENTS_FILE.read_text()   # kept so we can restore the file at the end

records = json.loads(_original_shipments)
print(f'Live shipment feed: {SHIPMENTS_FILE.resolve().name} - {len(records)} inbound shipments')
for r in records:
    print(f"  {r['shipment_id']}  {r['status']:<11} {r['units']:>4} units of {r['sku']} -> store {r['dest_store']}")

Live shipment feed: shipments.json - 3 inbound shipments
  SHP-88121  DELAYED       48 units of WM-KETTLE-01 -> store 4479
  SHP-88377  IN_TRANSIT    24 units of WM-AIRFRY-11 -> store 4479
  SHP-88402  DELIVERED    120 units of WM-TOWEL-04 -> store 2091


### The honest failure first

<div class='try-it'>
We ask the model &mdash; playing the ops assistant, exactly as deployed in E07 &mdash; the question a store manager asks every morning: <em>where is shipment SHP-88121?</em> The shipment exists; its live status sits in <code>shipments.json</code>. But the model has <strong>no way to reach it</strong>. Watch what comes back.
</div>

In [2]:
# A model with NO tools, asked about a live shipment it cannot possibly see.
q_ops = ('Where is inbound shipment SHP-88121 for store 4479 right now, '
         'and when will it actually arrive?')

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content':
         'You are the store operations assistant for a large retail chain. '
         'Answer the store manager directly and helpfully.'},
        {'role': 'user', 'content': q_ops}
    ]
)
print('STORE MANAGER:', q_ops)
print()
print('ASSISTANT (no tools):')
print(resp.choices[0].message.content)

STORE MANAGER: Where is inbound shipment SHP-88121 for store 4479 right now, and when will it actually arrive?

ASSISTANT (no tools):
I can help, but I don’t yet have access to your shipment tracking system from here.

To tell you **where SHP-88121 is right now** and **the actual arrival date/time for store 4479**, please paste one of the following (any single option works):

1) The **carrier/route + tracking number** shown for **SHP-88121**, or  
2) A screenshot/export from your **inbound tracking / WMS** page for **SHP-88121**, or  
3) The **bill of lading / PRO number** for that shipment.

Once you share that, I’ll return:
- current scan location/status (last checkpoint + timestamp),
- estimated delivery (and whether it’s updated),
- and the most likely **actual arrival window** for store **4479**.


<div class='where-box'>
<strong>WHERE it fails.</strong> The model has never seen shipment SHP-88121 &mdash; it literally cannot know. The only possible outputs are a made-up tracking story (a <strong>hallucination</strong> that would drive a real staffing decision at the store) or an admission that it has no access (a <strong>dead end</strong> &mdash; the manager gives up and phones the DC). Either way, the question is not answered.
</div>

<div class='why-box'>
<strong>WHY this matters to you.</strong> This is the exact failure behind most "our chatbot makes things up" incidents. It is not a model-quality problem &mdash; no amount of prompt engineering fixes <em>not having the data</em>. The fix is architectural: give the model a governed way to <strong>call your systems</strong>.
</div>

<hr class='section-divider'>

## Part 2 &nbsp;&middot;&nbsp; MCP From Scratch &mdash; a Server With Tools

<div class='topic-header'>
<strong>Two ideas, one standard.</strong> <em>Tool calling</em> is the model capability: instead of answering, the model can reply <em>"please run <code>track_shipment('SHP-88121')</code> and show me the result."</em> <strong>MCP (Model Context Protocol)</strong> is the open standard &mdash; introduced by Anthropic, now adopted across the industry &mdash; for packaging those tools so <em>any</em> AI application can discover and call them through one consistent plug. Think <strong>USB-C for AI</strong>: one port, many devices.
</div>

<div class='concept-box'>
<strong>The vocabulary (four words):</strong><br>
&bull; <strong>Host</strong> &mdash; the app the user talks to (Claude Desktop, an IDE, our notebook).<br>
&bull; <strong>Client</strong> &mdash; the connector inside the host that speaks the protocol.<br>
&bull; <strong>Server</strong> &mdash; the thing that <em>exposes</em> capabilities: your systems, wrapped.<br>
&bull; <strong>Tools &amp; resources</strong> &mdash; what a server offers: <em>tools</em> are functions the model can invoke; <em>resources</em> are live data it can read.
</div>

In [3]:
# Install the MCP libraries (quiet; safe to re-run).
%pip install -q fastmcp


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /Users/sudhanshusaxena/.venvs/jupyter/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<div class='try-it'>
<strong>Now we build a real MCP server</strong> for our supply-chain operations &mdash; in about 30 lines. Each tool is a plain Python function with a decorator. Note two things your teams will care about:<br>
&bull; the <strong>docstring and type hints are the contract</strong> &mdash; they become the published schema, the only thing the model ever sees;<br>
&bull; every function reads the file <em>inside</em> the call, so every answer is <strong>fresh by construction</strong> &mdash; no index to rebuild, ever.
</div>

In [4]:
from fastmcp import FastMCP, Client

mcp = FastMCP('RetailOpsServer')

def _load_shipments():
    return json.loads(SHIPMENTS_FILE.read_text())

@mcp.resource('file://shipments')
def all_shipments() -> str:
    """The full live inbound-shipment feed, re-read from disk on every call."""
    return SHIPMENTS_FILE.read_text()

@mcp.tool
def track_shipment(shipment_id: str) -> str:
    """Look up the live status, destination store, ETA and carrier note for one inbound shipment by its shipment_id (e.g. SHP-88121)."""
    for r in _load_shipments():
        if r['shipment_id'] == shipment_id.strip().upper():
            return json.dumps(r)
    return f'No shipment found with id {shipment_id}'

@mcp.tool
def check_store_stock(sku: str) -> str:
    """Return the current on-hand stock count for a SKU at store 4479 (e.g. WM-KETTLE-01, WM-AIRFRY-11, WM-TOWEL-04)."""
    stock = {'WM-KETTLE-01': 3, 'WM-AIRFRY-11': 17, 'WM-TOWEL-04': 41}
    s = stock.get(sku.strip().upper())
    return f'{sku}: {s} units on hand at store 4479' if s is not None else f'Unknown SKU {sku}'

@mcp.tool
def create_emergency_transfer(sku: str, units: int, dest_store: int) -> str:
    """Create an emergency stock transfer from the nearest DC to a store. THIS MOVES REAL INVENTORY AND COSTS MONEY - exposed only to approved workflows."""
    return (f'EMERGENCY TRANSFER QUEUED: {units} units of {sku} -> store {dest_store}. '
            'Regional manager approval required before dispatch.')

print('MCP server ready:', mcp.name)

MCP server ready: RetailOpsServer


<div class='concept-box'>
<strong>How we run it in this notebook:</strong> a production MCP server runs as its own process and the client connects over stdio or HTTP. Here we use FastMCP's <strong>in-memory transport</strong> &mdash; <code>Client(mcp)</code> wires the client straight to the server object in this same Python process. It is the <em>same real protocol and the same client code</em>; only the connection string changes when the server moves behind a network. Perfect for a classroom, zero subprocesses to babysit.
</div>

In [5]:
# Thin client helpers over the in-memory transport (Jupyter allows top-level await).
async def mcp_call(name, args):
    async with Client(mcp) as c:
        r = await c.call_tool(name, args)
        return r.content[0].text

async def mcp_read(uri):
    async with Client(mcp) as c:
        r = await c.read_resource(uri)
        return r[0].text

async def mcp_list_tools():
    async with Client(mcp) as c:
        return [(t.name, t.description, t.inputSchema) for t in await c.list_tools()]

print('Client helpers ready.')

Client helpers ready.


### Discovery &mdash; the server describes itself

<div class='try-it'>
This is the part that makes MCP more than "a folder of functions". A client &mdash; or an LLM &mdash; can ask any server: <em>what can you do?</em> No documentation, no source code, no meeting with the owning team. The catalog below is generated entirely from the docstrings and type hints you just saw.
</div>

In [7]:
for name, desc, schema in await mcp_list_tools():
    print('TOOL  :', name)
    print('  what:', desc)
    print('  args:', list(schema.get('properties', {}).keys()))
    print('-' * 64)

TOOL  : track_shipment
  what: Look up the live status, destination store, ETA and carrier note for one inbound shipment by its shipment_id (e.g. SHP-88121).
  args: ['shipment_id']
----------------------------------------------------------------
TOOL  : check_store_stock
  what: Return the current on-hand stock count for a SKU at store 4479 (e.g. WM-KETTLE-01, WM-AIRFRY-11, WM-TOWEL-04).
  args: ['sku']
----------------------------------------------------------------
TOOL  : create_emergency_transfer
  what: Create an emergency stock transfer from the nearest DC to a store. THIS MOVES REAL INVENTORY AND COSTS MONEY - exposed only to approved workflows.
  args: ['sku', 'units', 'dest_store']
----------------------------------------------------------------


In [8]:
# And tools really execute - this is real Python running on the server, not model text.
print(await mcp_call('track_shipment', {'shipment_id': 'SHP-88121'}))
print()
print(await mcp_call('check_store_stock', {'sku': 'WM-KETTLE-01'}))

{"shipment_id": "SHP-88121", "sku": "WM-KETTLE-01", "units": 48, "dest_store": 4479, "status": "DELAYED", "original_eta": "2026-08-11", "current_eta": "2026-08-18", "note": "Carrier weather hold at DC 6094; trailer re-slotted, +7 days."}

WM-KETTLE-01: 3 units on hand at store 4479


<div class='takeaway'>
<strong>Manager checkpoint.</strong> A tool result is <strong>computed, not generated</strong> &mdash; the JSON above came from disk, deterministically. When your teams say "the assistant is grounded", this is the strong version of that claim: numbers from systems, words from the model.
</div>

<hr class='section-divider'>

## Part 3 &nbsp;&middot;&nbsp; THE Demo &mdash; the World Changes, the Answer Follows

<div class='topic-header'>
The one demo that makes live tools click. We ask about a shipment, then <strong>the trailer arrives at the store</strong> &mdash; we update the shipment file, exactly as the transportation system would &mdash; and make the <em>same call again</em>. No re-indexing, no rebuild, nothing but a fresh read.
</div>

In [9]:
# Ask about SHP-88377 - currently in transit, due Aug 15.
print(await mcp_call('track_shipment', {'shipment_id': 'SHP-88377'}))

{"shipment_id": "SHP-88377", "sku": "WM-AIRFRY-11", "units": 24, "dest_store": 4479, "status": "IN_TRANSIT", "original_eta": "2026-08-15", "current_eta": "2026-08-15", "note": "On schedule."}


In [10]:
# The trailer arrives a day early. The transportation system updates the feed (that is ALL that changes).
data = json.loads(SHIPMENTS_FILE.read_text())
for r in data:
    if r['shipment_id'] == 'SHP-88377':
        r['status'] = 'DELIVERED'
        r['current_eta'] = '2026-08-14'
        r['note'] = 'Arrived a day early; received at store 4479, putaway in progress.'
SHIPMENTS_FILE.write_text(json.dumps(data, indent=2))
print('shipments.json updated: SHP-88377 is now DELIVERED.')

shipments.json updated: SHP-88377 is now DELIVERED.


In [11]:
# The SAME call again. Nothing was rebuilt. Watch the answer move.
print(await mcp_call('track_shipment', {'shipment_id': 'SHP-88377'}))

{"shipment_id": "SHP-88377", "sku": "WM-AIRFRY-11", "units": 24, "dest_store": 4479, "status": "DELIVERED", "original_eta": "2026-08-15", "current_eta": "2026-08-14", "note": "Arrived a day early; received at store 4479, putaway in progress."}


<div class='fix-box'>
<strong>Why the answer moved:</strong> <code>track_shipment</code> opens the file <em>inside the call</em>, so the second call simply read the newer truth. Contrast with E08's vector index or E09's report: both are snapshots that go stale until someone rebuilds them. <strong>Live data belongs behind a tool, not inside an index.</strong>
</div>

<hr class='section-divider'>

## Part 4 &nbsp;&middot;&nbsp; Now Let the MODEL Decide &mdash; the Tool-Calling Loop

<div class='topic-header'>
So far <em>we</em> called the tools by hand &mdash; useful for understanding, useless at scale. The real pattern: hand the model the <strong>tool catalog</strong>, and let <em>it</em> decide when to call what. The loop below is the beating heart of every tool-using assistant your teams will ever build &mdash; about 25 lines.
</div>

<div class='concept-box'>
<strong>One cycle in plain English:</strong> we send the question + the tool catalog &rarr; the model either answers, or replies <em>"call <code>track_shipment</code> with <code>{shipment_id: 'SHP-88121'}</code>"</em> &rarr; we execute that through MCP and append the result &rarr; the model reads it and writes the final answer. The model <strong>never executes anything itself</strong> &mdash; it only <em>requests</em>; our code runs the tools. That boundary is where all your governance lives.
</div>

In [12]:
# The bridge: MCP tool descriptions -> the schema the OpenAI API expects.
def mcp_to_openai(tools):
    return [{'type': 'function',
             'function': {'name': n, 'description': d or '', 'parameters': sch}}
            for (n, d, sch) in tools]

async def answer_with_tools(question, allowed=None, system_extra=''):
    """Tool-calling loop. `allowed` = names of MCP tools the model is given (None = all).
    Returns the answer plus a trace of every tool call actually made."""
    tools = [t for t in await mcp_list_tools() if allowed is None or t[0] in allowed]
    messages = [
        {'role': 'system', 'content':
         'You are the store operations assistant for a large retail chain. '
         'Use the provided tools for any live shipment or stock information - never guess it. '
         'If you have no tool that can answer or perform what is asked, say so plainly. ' + system_extra},
        {'role': 'user', 'content': question}]
    trace = []
    for _ in range(5):                                  # safety cap on loop rounds
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=mcp_to_openai(tools))
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return {'answer': msg.content, 'tool_calls': trace}
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = await mcp_call(tc.function.name, args)
            trace.append(f'{tc.function.name}({args})')
            messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
    return {'answer': '(loop cap reached)', 'tool_calls': trace}

print('Tool-calling loop ready.')

Tool-calling loop ready.


<div class='try-it'>
<strong>The rematch.</strong> The <em>identical</em> question that failed in Part 1 &mdash; same model, same question. The only change: the model now holds the tool catalog. We also print the <strong>trace</strong>, because a correct-sounding answer proves nothing &mdash; the trace proves the numbers came from the system, not from the model's imagination.
</div>

In [13]:
out = await answer_with_tools(q_ops, allowed=['track_shipment', 'check_store_stock'])
print('STORE MANAGER:', q_ops)
print()
print('Tools the model chose to call:', out['tool_calls'] or 'NONE')
print()
print('ASSISTANT (with tools):')
print(out['answer'])

STORE MANAGER: Where is inbound shipment SHP-88121 for store 4479 right now, and when will it actually arrive?

Tools the model chose to call: ["track_shipment({'shipment_id': 'SHP-88121'})"]

ASSISTANT (with tools):
Inbound shipment **SHP-88121** for **store 4479** is currently marked **DELAYED**.

- **Current location/status:** **Carrier weather hold at DC 6094** (with the trailer **re-slotted**)
- **Actual arrival (current ETA):** **2026-08-18**
- **Note:** “Carrier weather hold at DC 6094; trailer re-slotted, +7 days.”


<div class='fix-box'>
<strong>Same model, opposite outcome.</strong> In Part 1 this exact question hit a dead end &mdash; the model could only ask the manager to go fetch the tracking data by hand. Now the model saw <code>track_shipment</code> in its catalog, decided on its own to call it with <code>shipment_id='SHP-88121'</code>, received the live record (DELAYED &mdash; carrier weather hold at DC 6094, ETA slipped Aug 11 &rarr; Aug 18), and wrote a grounded reply. The intelligence chose; the plumbing executed; the trace lets you audit both.
</div>

<hr class='section-divider'>

## Part 5 &nbsp;&middot;&nbsp; Access Control &mdash; No Tool, No Power

<div class='topic-header'>
The question your security team will ask first: <em>"you gave an LLM access to production?"</em> The precise answer: the model can reach <strong>exactly the tools on the list we pass it &mdash; nothing else</strong>. That list is code we control, per assistant, per user, per session. Let us prove it, twice.
</div>

In [14]:
# Experiment 1: an assistant given ONLY track_shipment is asked a stock question.
out = await answer_with_tools('How many units of WM-KETTLE-01 are on hand at store 4479 right now?',
                              allowed=['track_shipment'])
print('Tools available to the model: track_shipment ONLY')
print('Tools it called:', out['tool_calls'] or 'NONE')
print()
print(out['answer'])

Tools available to the model: track_shipment ONLY
Tools it called: NONE

I can’t access store on-hand inventory with the tools available to me right now—I only have a shipment tracking lookup tool.

If you can provide an inventory/stock tool or a way to query on-hand for **WM-KETTLE-01** at **store 4479**, I can help immediately.


In [15]:
# Experiment 2: create_emergency_transfer EXISTS on the server - but this assistant is not given it.
out = await answer_with_tools(
    'Shipment SHP-88121 is badly delayed. Create an emergency transfer of 48 units '
    'of WM-KETTLE-01 to store 4479 right now.',
    allowed=['track_shipment', 'check_store_stock'])
print('Tools available: track_shipment, check_store_stock (create_emergency_transfer WITHHELD)')
print('Tools it called:', out['tool_calls'] or 'NONE')
print()
print(out['answer'])

Tools available: track_shipment, check_store_stock (create_emergency_transfer WITHHELD)
Tools it called: ["track_shipment({'shipment_id': 'SHP-88121'})"]

I checked shipment **SHP-88121**: it’s **DELAYED** due to a **carrier weather hold at DC 6094**; the trailer was re-slotted, for a **+7 day** impact (ETA updated to **2026-08-18**).

However, I **can’t create or place emergency transfers** with the tools I have available here (I can only **track shipments** and **check on-hand stock**). If you tell me what transfer system/workflow you use (e.g., item move request / WMS transfer order process), I can help draft the exact transfer request details for **48 units of WM-KETTLE-01** to **store 4479**.


<div class='takeaway'>
<strong>Least privilege, demonstrated.</strong> The stock count and the inventory-moving transfer both live one function away on the very same server &mdash; and the model could not touch either, because capability is granted <em>per tool, at the boundary we control</em>, not by what exists. This is the governance model to demand in every design review: read-only tools by default, money-moving tools behind approval workflows, and the allowed-list checked into code where it can be reviewed and audited.
</div>

<hr class='section-divider'>

## Part 6 &nbsp;&middot;&nbsp; RAG + Tools in One Answer &mdash; the Industry Pattern

<div class='topic-header'>
Real operational questions mix <strong>static knowledge</strong> (policy &mdash; what E08's RAG serves) with <strong>live state</strong> (this shipment, right now &mdash; what tools serve). Consider: <em>"SHP-88121 is late &mdash; what does our playbook say I should do, and where is it?"</em> Policy alone cannot say where the trailer is; the tracker alone cannot say what the playbook requires. The shipped pattern uses both in one answer.
</div>

In [16]:
# Static knowledge: three ops-policy snippets. (Stand-in for E08's vector store -
# same role; retrieval kept deliberately simple here so the focus stays on the combination.)
POLICY = {
 'inbound_delay': ('Inbound delay playbook: if a replenishment shipment slips more than 5 days past its '
   'original ETA, the receiving store must request an emergency transfer from the nearest DC '
   'and flag the SKU for substitution on the planogram until stock recovers.'),
 'receiving': ('Receiving policy: delivered trailers must complete putaway within 24 hours; '
   'discrepancies over 2% of units must be logged in the claims system within 48 hours.'),
 'returns_window': ('Customer returns policy: most items may be returned within 30 days; '
   'electronics within 15 days, unopened or defective.')}

def retrieve_policy(question):
    """Toy retriever: score snippets by keyword overlap, return the best one (E08 did this with embeddings)."""
    qw = set(question.lower().split())
    return max(POLICY.values(), key=lambda s: len(qw & set(s.lower().split())))

combo_q = ('Inbound replenishment shipment SHP-88121 is late - what does our playbook require me to do '
           'about the delay, and where is the shipment right now?')
print('Retrieved policy context:\n ', retrieve_policy(combo_q))

Retrieved policy context:
  Inbound delay playbook: if a replenishment shipment slips more than 5 days past its original ETA, the receiving store must request an emergency transfer from the nearest DC and flag the SKU for substitution on the planogram until stock recovers.


In [17]:
# One answer, two knowledge sources: retrieved policy (static) + shipment tool (live).
out = await answer_with_tools(
    combo_q,
    allowed=['track_shipment'],
    system_extra='Company policy context you may rely on: ' + retrieve_policy(combo_q))
print('Tools it called:', out['tool_calls'] or 'NONE')
print()
print(out['answer'])

Tools it called: ["track_shipment({'shipment_id': 'SHP-88121'})"]

### Where the shipment is right now
- **Shipment:** SHP-88121  
- **Status:** **DELAYED**  
- **Destination store:** **4479**  
- **Original ETA:** **2026-08-11**  
- **Current ETA:** **2026-08-18** (**+7 days**)  
- **Carrier note:** *Carrier weather hold at DC 6094; trailer re-slotted, +7 days.*

### What the inbound delay playbook requires (since it’s >5 days late)
Because the shipment is slipping **more than 5 days past the original ETA** (**+7 days**), the playbook requires you to:

1. **Request an emergency transfer**  
   - **From the nearest DC** to support store 4479.

2. **Flag the SKU for substitution on the planogram until stock recovers**  
   - **SKU on this shipment:** **WM-KETTLE-01**

If you tell me which DC is “nearest” per your store setup (or your DC list), I can help you format the exact emergency transfer request details you’ll submit.


<div class='takeaway'>
<strong>This is the pattern you ship.</strong> Retrieval supplied the <em>rule</em> (&gt;5-day slip &rarr; emergency transfer + substitution flag &mdash; static), the tool supplied the <em>fact</em> (SHP-88121 delayed 7 days, weather hold at DC 6094 &mdash; live), and the model wove them into one accountable answer. Neither source alone could do it.
</div>

<div class='concept-box'>
<strong>The routing rule to memorise</strong> &mdash; the question is never "is this data complex?", it is <strong>"is this data live or static?"</strong>
</div>

<table class='compare-table'>
<tr><th>Dimension</th><th>RAG (E08)</th><th>SQL analytics (E09)</th><th>Tools / MCP (E10)</th></tr>
<tr><td><strong>Best for</strong></td><td>Policies, manuals, documents</td><td>Aggregates over business data</td><td>Live state &amp; actions</td></tr>
<tr><td><strong>Freshness</strong></td><td>Frozen at indexing time</td><td>As fresh as the last load</td><td>Read at question time</td></tr>
<tr><td><strong>Can it act?</strong></td><td>No</td><td>No</td><td><strong>Yes</strong> &mdash; and that is exactly why it needs governance</td></tr>
<tr><td><strong>Failure smell</strong></td><td>Confidently stale answers</td><td>Right answer, yesterday's data</td><td>Wrong tool, or a tool it should never have had</td></tr>
</table>

<hr class='section-divider'>

## Wrap-Up &mdash; What Was Built in This Session

<table class='compare-table'>
<tr><th>Part</th><th>What happened</th><th>The lesson</th></tr>
<tr><td>1</td><td>Model asked about a live shipment, no tools</td><td>No access = hallucination or dead end; not fixable by prompting</td></tr>
<tr><td>2</td><td>MCP server: 3 tools + 1 resource in ~30 lines</td><td>Docstrings + type hints ARE the contract; discovery replaces documentation</td></tr>
<tr><td>3</td><td>File changed &rarr; same call, new answer</td><td>Tools read at question time; live data never belongs in an index</td></tr>
<tr><td>4</td><td>The tool-calling loop; the Part-1 question, answered</td><td>Model requests, our code executes; the trace makes it auditable</td></tr>
<tr><td>5</td><td>Withheld tools &rarr; model powerless</td><td>Least privilege is a per-tool allowed-list you control in code</td></tr>
<tr><td>6</td><td>Policy (RAG) + shipment (tool) in one answer</td><td>Route by live-vs-static; real assistants use both</td></tr>
</table>

<div class='takeaway'>
<strong>Key Takeaways for Engineering Managers</strong>
<ol>
<li><strong>Tool calling is a request, not an execution</strong> &mdash; the model asks; your code runs the function. Every control you need lives at that boundary.</li>
<li><strong>MCP is USB-C for AI</strong> &mdash; wrap a system once, and any compliant host can discover and use it. Ask vendors "do you ship an MCP server?" the way you ask "do you have an API?"</li>
<li><strong>Live vs static decides the architecture</strong> &mdash; documents &rarr; RAG, aggregates &rarr; SQL, current state &amp; actions &rarr; tools.</li>
<li><strong>The tool list is your security perimeter</strong> &mdash; least privilege per assistant, inventory- and money-moving tools behind approvals, allowed-lists in reviewed code.</li>
<li><strong>Demand the trace</strong> &mdash; a grounded-sounding answer without a tool-call trace is unverifiable. Logging tool calls is non-negotiable in production.</li>
<li><strong>The schema is the docstring</strong> &mdash; tool descriptions are prompt engineering; vague docstrings produce wrong tool choices.</li>
</ol>
</div>

In [ ]:
# ── Housekeeping: restore shipments.json to its original state so this notebook re-runs cleanly.
SHIPMENTS_FILE.write_text(_original_shipments)
print('shipments.json restored to its original snapshot.')

<div class='topic-header'>
<strong>&#128279; The gap we leave &mdash; and where E11 begins</strong><br><br>
Look back at what just happened, honestly: the model could call tools &mdash; but <em>we</em> orchestrated every step. We chose which question to ask, wired one retrieval, capped the loop at a handful of rounds, and stitched the story together cell by cell. That is fine for one question; nobody wants to hand-drive a workflow of twenty steps &mdash; "check the shipment, if it is delayed find the playbook, draft the store notice, request the transfer, log the case."<br><br>
<strong>E11 &mdash; Agents</strong> closes exactly that gap: give the model a <em>goal</em> instead of a question, and let it <strong>plan its own sequence of tool calls</strong> &mdash; deciding what to look up, in what order, and when it is done &mdash; while our governance boundary from Part 5 keeps it on the rails.
</div>